In [ ]:
import json
import pandas as pd
from pathlib import Path

# ✅ Sesuaikan path ini
BASE = 'C:/Users/Cerdas05/Skripshot/U-Net-Architecture/checkpoints (50 ep 30 May 2026)'

rows = []
for folder in sorted(Path(BASE).iterdir()):
    hist_path = folder / 'history.json'
    if not hist_path.exists():
        continue

    with open(hist_path) as f:
        h = json.load(f)

    val      = h.get('val', [])
    if not val:
        continue

    best_idx = max(range(len(val)), key=lambda i: val[i]['dice'])
    best     = val[best_idx]

    # ✅ Parse nama folder: original_adamw_lr0.0001
    name  = folder.name                    # original_adamw_lr0.0001
    parts = name.split('_')               # ['original', 'adamw', 'lr0.0001']

    skenario  = parts[0].capitalize()     # Original
    optimizer = parts[1].upper()          # ADAMW
    lr        = parts[2].replace('lr','') # 0.0001

    rows.append({
        'Skenario'       : skenario,
        'Optimizer'      : optimizer,
        'LR'             : lr,
        'Best Epoch'     : best_idx + 1,
        'Total Epoch'    : len(val),
        'Dice (%)'       : round(best['dice']        * 100, 2),
        'IoU (%)'        : round(best['iou']         * 100, 2),
        'Accuracy (%)'   : round(best['accuracy']    * 100, 2),
        'Precision (%)'  : round(best['precision']   * 100, 2),
        'Recall (%)'     : round(best['recall']      * 100, 2),
        'Specificity (%)': round(best['specificity'] * 100, 2),
        'Loss'           : round(best['loss'], 4),
    })

df = pd.DataFrame(rows)
df = df.sort_values(['Skenario', 'Optimizer', 'LR'])

print(df.to_string(index=False))

# Simpan ke CSV
df.to_csv(f'{BASE}/hasil_18_eksperimen.csv', index=False)
print('\n✅ Disimpan: hasil_18_eksperimen.csv')